# 01 — Data Acquisition
**Project:** Phase-Field Informed ML for Microstructure Prediction in Ti-Nb-O Refractory Alloys  
**Author:** Anosike Kelechi Kenneth  
**Purpose:** Load the matbench-mp-gap dataset, sample a reproducible 20k subset, and save it for downstream notebooks.  
**Run order:** Run this notebook FIRST before any other notebook.

---

In [ ]:
# Section 1: Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matminer.datasets import load_dataset

print('All imports OK')

In [ ]:
# Section 2: Load the matbench-mp-gap dataset
# Source: Materials Project via matminer
# Citation: Dunn et al., npj Computational Materials 6, 138 (2020)
# No API key required - matminer handles download automatically

print('Loading matbench-mp-gap dataset...')
df_full = load_dataset('matbench_mp_gap')
print(f'Full dataset shape: {df_full.shape}')
print(f'Columns: {df_full.columns.tolist()}')
print(f'\nFirst few rows:')
df_full.head()

In [ ]:
# Section 3: Sample reproducible 20k subset
# Using random_state=42 for full reproducibility across runs
RANDOM_STATE = 42
SAMPLE_SIZE = 20000

df = df_full.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Subset shape: {df.shape}')
print(f'\nTarget property (gap pbe) statistics:')
print(df['gap pbe'].describe().round(4))

In [ ]:
# Section 4: Basic data quality check
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nNumber of metals (bandgap = 0): {(df["gap pbe"] == 0).sum()}')
print(f'Number of insulators/semiconductors (bandgap > 0): {(df["gap pbe"] > 0).sum()}')
print(f'\nClass balance: {(df["gap pbe"] == 0).sum()/len(df)*100:.1f}% metals, {(df["gap pbe"] > 0).sum()/len(df)*100:.1f}% non-metals')

In [ ]:
# Section 5: Target property distribution plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Raw distribution
axes[0].hist(df['gap pbe'], bins=60, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_xlabel('Band Gap (eV)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Band Gap Distribution (20k subset)', fontsize=12)
axes[0].axvline(df['gap pbe'].mean(), color='red', linestyle='--', label=f'Mean={df["gap pbe"].mean():.2f} eV')
axes[0].legend()

# Non-zero only
nonzero = df[df['gap pbe'] > 0]['gap pbe']
axes[1].hist(nonzero, bins=60, color='teal', alpha=0.8, edgecolor='white')
axes[1].set_xlabel('Band Gap (eV)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Band Gap Distribution (non-metals only)', fontsize=12)

plt.suptitle('matbench-mp-gap: Target Property Overview', fontsize=13)
plt.tight_layout()
plt.savefig('../figures/01_bandgap_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to figures/01_bandgap_distribution.png')

In [ ]:
# Section 6: Extract composition strings and save subset
# Convert pymatgen Structure to composition string for downstream featurization
print('Extracting composition strings from structures...')
df['composition_str'] = df['structure'].apply(lambda s: str(s.composition))
df['formula'] = df['structure'].apply(lambda s: s.composition.reduced_formula)

print(f'Sample compositions:')
print(df[['formula', 'gap pbe']].head(10).to_string(index=False))

# Save the subset dataframe index for reproducibility
# (full dataframe with structures is too large to save as CSV)
df[['formula', 'composition_str', 'gap pbe']].to_csv('../data/bandgap_subset_20k.csv', index=False)
print('\nSaved: data/bandgap_subset_20k.csv')
print(f'Shape: {df[["formula", "composition_str", "gap pbe"]].shape}')

In [ ]:
# Section 7: Summary
print('=== Data Acquisition Summary ===')
print(f'Dataset: matbench-mp-gap')
print(f'Full size: {df_full.shape[0]:,} compounds')
print(f'Working subset: {len(df):,} compounds (random_state={RANDOM_STATE})')
print(f'Target: Band gap energy (eV) via GGA-PBE DFT')
print(f'Mean bandgap: {df["gap pbe"].mean():.3f} eV')
print(f'Std bandgap: {df["gap pbe"].std():.3f} eV')
print(f'Metals (gap=0): {(df["gap pbe"]==0).sum()} ({(df["gap pbe"]==0).mean()*100:.1f}%)')
print(f'Saved to: data/bandgap_subset_20k.csv')
print('\nNext step: Run 02_eda_featurization.ipynb')